In [1]:
from ucimlrepo import fetch_ucirepo

heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

print(X.shape)
print(X.head())

(303, 13)
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

    ca  thal  
0  0.0   6.0  
1  3.0   3.0  
2  2.0   7.0  
3  0.0   3.0  
4  0.0   3.0  


In [2]:
print(y.value_counts())
print()
print(X.isnull().sum())

num
0      164
1       55
2       36
3       35
4       13
Name: count, dtype: int64

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
dtype: int64


In [3]:
import pandas as pd

df = pd.concat([X, y], axis=1)

df['target'] = (df['num'] > 0).astype(int)
df = df.drop(columns=['num'])

df = df.dropna()

print(df.shape)
print(df['target'].value_counts())

(297, 14)
target
0    160
1    137
Name: count, dtype: int64


In [14]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, URL

load_dotenv(".env", override=True)

url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(url)

df.to_sql("patients", engine, if_exists="replace", index=False)

print("Loaded into PostgreSQL!")

Loaded into PostgreSQL!


In [15]:
check = pd.read_sql("SELECT * FROM patients LIMIT 5", engine)
print(check)

count = pd.read_sql("SELECT target, COUNT(*) AS patients FROM patients GROUP BY target", engine)
print(count)

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

    ca  thal  target  
0  0.0   6.0       0  
1  3.0   3.0       1  
2  2.0   7.0       1  
3  0.0   3.0       0  
4  0.0   3.0       0  
   target  patients
0       0       160
1       1       137
